<a href="https://colab.research.google.com/github/yeshaa23/Project-A-Kelompok-4-Pertamina-PBAGenap/blob/main/Scraping_Artikel_Pertamina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scrapping Artikel Pertamina

## Install dan Import Library

In [1]:
!pip install newspaper3k requests beautifulsoup4 pandas selenium

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 19.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 16.7 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=c0e9f2cd1c4077631ac4ea30f287dae3774eb94c374a96d7bca38ab3625d7820
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created

In [2]:
!pip install lxml[html-clean]

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load Dataset

In [5]:
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()

Saving Kelompok 4 Link Artikel Pertamina - Scrap .csv to Kelompok 4 Link Artikel Pertamina - Scrap .csv


In [6]:
df = pd.read_csv(io.BytesIO(uploaded['Kelompok 4 Link Artikel Pertamina - Scrap .csv']))
df.head()

,No,Link,Sentiment,Penerbit,Tag
0,1,https://money.kompas.com/image/2017/10/03/1815...,Netral,money.kompas.com,Ekonomi
1,2,https://money.kompas.com/read/2016/06/06/14280...,Positif,money.kompas.com,Kecelakaan Kerja
2,3,https://money.kompas.com/read/2017/02/03/12195...,Positif,money.kompas.com,Migas
3,4,https://money.kompas.com/read/2016/05/27/20434...,Positif,money.kompas.com,Gangguan Operasional
4,5,https://money.kompas.com/read/2017/03/24/11200...,Positif,money.kompas.com,Migas


## Scraping Artikel
* Fungsi untuk memeriksa status URL
* Fungsi untuk scraping artikel dengan BeautifulSoup
* Fungsi untuk scraping artikel menggunakan newspaper3k
* Scraping semua link berita

In [11]:
import requests

# Function untuk memeriksa status URL
def check_url(url):
    try:
        response = requests.get(url, timeout=10)
        return response.status_code == 200
    except requests.RequestException:
        return False  # URL tidak bisa diakses

In [12]:
# Fungsi untuk scraping artikel dengan BeautifulSoup
def scrape_article(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            return "Gagal", f"Status Code: {response.status_code}", "request_gagal"

        soup = BeautifulSoup(response.content, 'html.parser')

        # Mapping sumber berita dan tag HTML yang sesuai
        source_mapping = {
            'detik.com': {'title': 'h1', 'content': 'div.detail__body-text'},
            'kompas.com': {'title': 'h1.article-title', 'content': 'div.article-content'},
            'liputan6.com': {'title': 'h1.article-title', 'content': 'div.article-body__content'},
            'tempo.co': {'title': 'h1.article-title', 'content': 'div.article-body'},
            'bisnis.com': {'title': 'h1', 'content': 'div.article-body'},
            'kumparan.com': {'title': 'h1', 'content': 'div.article-body'},
            'republika.co.id': {'title': 'h1', 'content': 'div.article-content'},
            'cnbcindonesia.com': {'title': 'h1', 'content': 'div.article-body'},
            'indef.or.id': {'title': 'h1', 'content': 'div.article-body'},
            'setkab.go.id': {'title': 'h1', 'content': 'div.article-body'}
        }

        # Tentukan situs berdasarkan URL
        for source, tags in source_mapping.items():
            if source in url:
                title_tag = soup.find(tags['title'])
                article_body = soup.find(tags['content'])

                title = title_tag.get_text(strip=True) if title_tag else "Judul tidak ditemukan"
                content = "\n".join([p.get_text(strip=True) for p in article_body.find_all('p')]) if article_body else "Konten tidak ditemukan"
                return title, content.strip(), "success"

        return "Gagal", "Tidak dikenali sumber berita", "source_unknown"

    except Exception as e:
        return "Gagal", f"Error: {str(e)}", "exception"

In [13]:
# Fungsi untuk scraping artikel menggunakan newspaper3k
from newspaper import Article

def scrape_article_with_newspaper(url):
    article = Article(url, language='id')  # Tentukan bahasa 'id' untuk Bahasa Indonesia
    article.download()
    article.parse()

    # Mendapatkan artikel dan informasi terkait
    title = article.title
    content = article.text
    return title, content, "success"

In [14]:
import time

# Proses scraping untuk seluruh link berita yang ada di CSV
results = []
failed_results = []  # Menyimpan link yang gagal di-scrape

total_links = len(df)

# Menyimpan jumlah berhasil dan gagal
successful_count = 0
failed_count = 0

for index, row in df.iterrows():
    link = row['Link']  # Pastikan kolom 'Link' ada di file CSV kamu
    tag = row['Tag']  # Menambahkan kolom tag dari CSV awal
    sentiment = row['Sentiment']  # Menambahkan kolom sentimen dari CSV awal
    publisher = row['Penerbit']  # Menambahkan kolom penerbit dari CSV awal

    if pd.notna(link) and link.startswith('http'):
        print(f"Scraping ({index + 1}/{total_links}): {link}")

        # Cek apakah URL bisa diakses
        if not check_url(link):
            failed_results.append({'Link': link, 'Judul': 'URL tidak valid', 'Isi Berita': '', 'Status': 'failed', 'Tag': tag, 'Sentimen': sentiment, 'Penerbit': publisher})
            failed_count += 1  # Menambah hitungan gagal
            continue  # Skip link ini jika tidak bisa diakses

        # Coba scraping dengan newspaper3k terlebih dahulu
        try:
            title, content, status = scrape_article_with_newspaper(link)
        except Exception:
            # Jika gagal dengan newspaper3k, coba dengan BeautifulSoup dan requests
            title, content, status = scrape_article(link)

        # Memeriksa jika scraping gagal (judul dan konten kosong)
        if title == "Judul tidak ditemukan" or content == "Konten tidak ditemukan":
            status = "error"  # Memastikan status 'error' jika judul atau konten tidak ditemukan
            failed_results.append({'Link': link, 'Judul': title, 'Isi Berita': content, 'Status': status, 'Tag': tag, 'Sentimen': sentiment, 'Penerbit': publisher})
            failed_count += 1  # Menambah hitungan gagal
            print(f"Link gagal di-scrape: {link}")
        else:
            results.append({'Link': link, 'Judul': title, 'Isi Berita': content, 'Status': 'success', 'Tag': tag, 'Sentimen': sentiment, 'Penerbit': publisher})
            successful_count += 1  # Menambah hitungan berhasil

        time.sleep(1)  # 1 detik jeda antar request
    else:
        results.append({'Link': link, 'Judul': 'Tidak ada link', 'Isi Berita': '', 'Status': 'link_kosong', 'Tag': tag, 'Sentimen': sentiment, 'Penerbit': publisher})
        failed_count += 1  # Menambah hitungan gagal

# Menampilkan jumlah artikel yang berhasil dan gagal
print(f"Berhasil di-scrape: {successful_count} artikel")
print(f"Gagal di-scrape: {failed_count} artikel")

Scraping (1/2500): https://money.kompas.com/image/2017/10/03/181500926/csr-pertamina-lubricants-kini-fokus-ke-penciptaan-kemandirian-ekonomi
Scraping (2/2500): https://money.kompas.com/read/2016/06/06/142802426/gandeng.dua.bank.pertamina.lubricants.yakin.penjualan.pelumas.terdongkrak.
Scraping (3/2500): https://money.kompas.com/read/2017/02/03/121955826/dua.pucuk.pimpinan.pertamina.dicopot.yenni.andayani.jabat.plt.dirut
Scraping (4/2500): https://money.kompas.com/read/2016/05/27/204344726/ini.yang.dilakukan.ptkam.untuk.efisiensi.oil.losses.pertamina
Scraping (5/2500): https://money.kompas.com/read/2017/03/24/112000626/ini.tugas.pertama.adiatma.sardjito.sebagai.jubir.baru.pertamina
Scraping (6/2500): https://money.kompas.com/read/2017/01/16/153000226/akhirnya.proyek.kilang.tuban.gunakan.lahan.klhk.di.tanjung.awar-awar
Scraping (7/2500): https://nasional.kompas.com/read/2017/11/03/15521211/jadi-tersangka-dana-pensiun-pertamina-edward-soeryadjaya-dicegah-ke-luar
Scraping (8/2500): https:/

## Hasil scraping
menyimpan hasil di drive

In [15]:
# Membuat DataFrame dari data hasil scraping
df_results = pd.DataFrame(results)
print(df_results.head())

                                                Link  \
0  https://money.kompas.com/image/2017/10/03/1815...   
1  https://money.kompas.com/read/2016/06/06/14280...   
2  https://money.kompas.com/image/2017/08/03/0900...   
3  https://money.kompas.com/read/2016/04/11/17345...   
4  https://money.kompas.com/image/2017/11/27/1656...   

                                               Judul  \
0  Foto : CSR Pertamina Lubricants Kini Fokus ke ...   
1  Gandeng Dua Bank, Pertamina Lubricants Yakin P...   
2  Foto : Cegah Kepunahan, Pertamina Lestarikan T...   
3  Pertamina Menaikkan Ongkos Angkut Minyak Penam...   
4  Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...   

                                          Isi Berita   Status  \
0  Diskusi mengenai CSR di industri pelumas berta...  success   
1  JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...  success   
2  Pelepasan Indukan Tuntong Laut oleh tim konser...  success   
3  BOJONEGORO, KOMPAS.com - Pemerintah terus meng...  success   
4

In [16]:
# Menyimpan hasil scraping dalam CSV di Google Drive
output_path = '/content/drive/MyDrive/ProjectA-PBA/hasil_scraping_berita.csv'
df_results.to_csv(output_path, index=False, encoding='utf-8')